# Generate AI Research Proposals - Baseline Condition

This notebook implements the baseline condition for AI-generated research proposals as outlined in the analysis plan.

## Steps:
1. Generate 23 research ideas from each AI model (GPT, Gemini, Claude) using the `generate_ideas_baseline` prompt
2. Save all titles and abstracts to a CSV file in `data/ai-proposals/baseline`
3. For each idea, generate a full proposal using the `generate_proposals` prompt with the same model
4. Add full proposals to the CSV file

## Setup and Install Dependencies

First, ensure all required packages are installed from `src/requirements.txt`.

In [ ]:
# Option 1: Run setup.py (recommended - also creates .env config file)
# This uses the setup script from src/setup.py which installs from src/requirements.txt
import subprocess
import sys
from pathlib import Path


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')


PROJECT_ROOT = find_project_root()
REQUIREMENTS_PATH = PROJECT_ROOT / 'src' / 'requirements.txt'
SETUP_SCRIPT_PATH = PROJECT_ROOT / 'src' / 'setup.py'

# Uncomment ONE of the following options:

# Option A: Run the full setup script (installs dependencies + creates .env file)
# subprocess.check_call([sys.executable, str(SETUP_SCRIPT_PATH)], cwd=PROJECT_ROOT)

# Option B: Just install requirements from src/requirements.txt
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH)], cwd=PROJECT_ROOT)

print('ℹ️  To install dependencies, uncomment one of the options above and run this cell')
print(f'✓ Dependencies defined in: {REQUIREMENTS_PATH}')
print(f'✓ Setup script available at: {SETUP_SCRIPT_PATH}')


## Import Required Modules

In [ ]:
import sys
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
import logging


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')


PROJECT_ROOT = find_project_root()

# Add src to path to import custom modules
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import custom modules from src/
from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print('✓ Imports successful')
print(f'✓ Working directory: {os.getcwd()}')
print(f'✓ Project root: {PROJECT_ROOT}')
print(f'✓ Python path includes: {src_path}')


## Load Configuration and Data

In [ ]:
# Load call and information about NCEMS
call_and_info_path = PROJECT_ROOT / 'data' / 'call_and_info.json'
with open(call_and_info_path, 'r') as f:
    call_and_info = json.load(f)

research_call = call_and_info['call']
ncems_info = call_and_info['info']

print('✓ Loaded call and NCEMS information')
print(f'✓ Source file: {call_and_info_path}')
print(f'\nResearch call preview: {research_call[:200]}...')
print(f'\nNCEMS info preview: {ncems_info[:200]}...')


In [ ]:
# =========================
# Condition Configuration
# =========================
# Set this once at the top of the notebook.
CONDITION = 'how_to_think'  # e.g., 'minimal', 'baseline', 'how_to_think', 'persona', 'rephrased/minimal'
GENERATE_NEW_IDEAS = False  # True => run idea generation; False => load existing idea files for CONDITION

# Prompt/template choices
IDEA_PROMPT_TEMPLATE = 'generate_ideas_minimal'
PROPOSAL_PROMPT_TEMPLATE = 'generate_proposals_minimal'

# Idea generation count (only used when GENERATE_NEW_IDEAS=True)
NUM_IDEAS_PER_MODEL = 23

# Optional explicit file override (set to a path string or keep None)
IDEAS_FILE_OVERRIDE = None

# Optional glob filter inside condition dir (keep None to auto-detect all csv/json/xlsx/xls)
IDEAS_GLOB = None

condition_slug = CONDITION.replace('/', '_').replace(' ', '_')
ideas_input_dir = PROJECT_ROOT / 'data' / 'ai-proposals' / CONDITION
output_dir = ideas_input_dir
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

MODEL_CANONICAL_MAP = {
    'gpt-5.2': 'gpt-5.2',
    'gpt5.2': 'gpt-5.2',
    'gpt': 'gpt-5.2',
    'gpt-5': 'gpt-5.2',
    'gemini-3-pro-preview': 'gemini-3-pro-preview',
    'gemini3propreview': 'gemini-3-pro-preview',
    'gemini': 'gemini-3-pro-preview',
    'claude-opus-4-5': 'claude-opus-4-5',
    'claudeopus45': 'claude-opus-4-5',
    'claude': 'claude-opus-4-5',
}


def _normalize_key(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


def canonicalize_model(model_value):
    if model_value is None:
        return None
    key = _normalize_key(model_value)
    return MODEL_CANONICAL_MAP.get(key, str(model_value))


def infer_model_from_filename(path: Path):
    name = path.name.lower()
    if 'claude' in name:
        return 'claude-opus-4-5'
    if 'gemini' in name:
        return 'gemini-3-pro-preview'
    if 'gpt' in name:
        return 'gpt-5.2'
    return None


def _extract_ideas_from_json_obj(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for k in ['research_ideas', 'ideas', 'records', 'data']:
            if k in obj and isinstance(obj[k], list):
                return obj[k]
        # fallback: single record dict
        if any(k in obj for k in ['title', 'abstract']):
            return [obj]
    return []


def _load_records_from_file(path: Path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path).to_dict('records')
    if suffix in ('.xlsx', '.xls'):
        return pd.read_excel(path).to_dict('records')
    if suffix == '.json':
        with open(path, 'r') as f:
            obj = json.load(f)
        return _extract_ideas_from_json_obj(obj)
    return []


def _pick_value(rec, candidates):
    # exact key first
    for c in candidates:
        if c in rec and pd.notna(rec[c]):
            val = rec[c]
            if str(val).strip() != '':
                return val
    # normalized key fallback
    norm = {_normalize_key(k): k for k in rec.keys()}
    for c in candidates:
        ck = _normalize_key(c)
        if ck in norm:
            v = rec[norm[ck]]
            if pd.notna(v) and str(v).strip() != '':
                return v
    return None


def normalize_idea_records(records, source_file: Path):
    out = []
    model_from_file = infer_model_from_filename(source_file)
    for rec in records:
        if not isinstance(rec, dict):
            continue

        title = _pick_value(rec, ['title', 'idea_title', 'proposal_title', 'research_title'])
        abstract = _pick_value(rec, ['abstract', 'summary', 'description'])
        author = _pick_value(rec, ['author', 'persona_author', 'scientist_author', 'author_name', 'target_author'])
        model_raw = _pick_value(rec, ['model', 'llm_model', 'generator_model', 'ai_model'])

        model = canonicalize_model(model_raw) if model_raw is not None else model_from_file
        if model is None:
            # keep row but mark unknown; generation loop will skip safely
            model = ''

        if title is None or abstract is None:
            continue

        out.append({
            'model': model,
            'author': author if author is not None else model,
            'title': str(title).strip(),
            'abstract': str(abstract).strip(),
            'generated_at': _pick_value(rec, ['generated_at', 'created_at', 'timestamp']),
            'source_file': source_file.name,
        })
    return out


def get_condition_idea_files():
    if IDEAS_FILE_OVERRIDE:
        file_path = Path(IDEAS_FILE_OVERRIDE)
        if not file_path.exists():
            raise FileNotFoundError(f'IDEAS_FILE_OVERRIDE not found: {file_path}')
        return [file_path]

    if not ideas_input_dir.exists():
        raise FileNotFoundError(f'Condition folder not found: {ideas_input_dir}')

    if IDEAS_GLOB:
        files = sorted(ideas_input_dir.glob(IDEAS_GLOB))
    else:
        files = [
            f for f in sorted(ideas_input_dir.iterdir())
            if f.is_file() and f.suffix.lower() in ('.csv', '.json', '.xlsx', '.xls')
        ]

    if not files:
        raise FileNotFoundError(
            f'No idea files found in {ideas_input_dir} '
            f'(glob={IDEAS_GLOB!r}, override={IDEAS_FILE_OVERRIDE!r})'
        )

    return files


def load_ideas_for_condition():
    files = get_condition_idea_files()
    normalized = []
    for fp in files:
        records = _load_records_from_file(fp)
        part = normalize_idea_records(records, fp)
        normalized.extend(part)
        print(f'Loaded {len(part):3d} ideas from {fp.name}')

    ideas_df = pd.DataFrame(normalized)
    if ideas_df.empty:
        raise ValueError('No valid ideas loaded after parsing.')

    # clean and basic validation
    ideas_df = ideas_df.dropna(subset=['title', 'abstract']).copy()
    ideas_df['title'] = ideas_df['title'].astype(str).str.strip()
    ideas_df['abstract'] = ideas_df['abstract'].astype(str).str.strip()
    ideas_df = ideas_df[(ideas_df['title'] != '') & (ideas_df['abstract'] != '')].reset_index(drop=True)

    # order columns
    for c in ['model', 'author', 'title', 'abstract', 'generated_at', 'source_file']:
        if c not in ideas_df.columns:
            ideas_df[c] = ''
    ideas_df = ideas_df[['model', 'author', 'title', 'abstract', 'generated_at', 'source_file']]

    print(f'\nParsed total ideas: {len(ideas_df)}')
    print('Ideas per model:')
    print(ideas_df['model'].value_counts(dropna=False))
    if 'author' in ideas_df.columns:
        print('\nTop author labels:')
        print(ideas_df['author'].fillna('').value_counts().head(10))

    return ideas_df


print('✓ Condition configuration loaded')
print(f'  CONDITION             : {CONDITION}')
print(f'  GENERATE_NEW_IDEAS    : {GENERATE_NEW_IDEAS}')
print(f'  ideas_input_dir       : {ideas_input_dir}')
print(f'  output_dir            : {output_dir}')
print(f'  condition_slug        : {condition_slug}')


## Initialize AI Models Interface

In [ ]:
# Initialize AI interface (will load API keys from config.env or .env file)
# The setup.py script creates .env in the root directory
ai_interface = AIModelsInterface(config_path='.env')

# Get available models
available_models = ai_interface.get_available_models()
print(f"✓ Available models: {available_models}")

# Define the models we want to use (GPT, Gemini, Claude)
models_to_use = ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']
models_to_use = [m for m in models_to_use if m in available_models]
print(f"\nUsing models: {models_to_use}")

## Step 1: Build Ideas Table (Generate New or Load Existing)

- If `GENERATE_NEW_IDEAS=True`, generate ideas from each model using the configured prompt.
- If `GENERATE_NEW_IDEAS=False`, load existing idea files from `data/ai-proposals/{CONDITION}`.
- Supported input formats: `.csv`, `.json`, `.xlsx`/`.xls`.
- Parsed canonical fields used for proposal generation: `model`, `author`, `title`, `abstract`.


In [ ]:
# Initialize prompt manager
prompt_manager = PromptManager()

if GENERATE_NEW_IDEAS:
    ideas_prompt_template = prompt_manager.get_template(IDEA_PROMPT_TEMPLATE)
    ideas_prompt = ideas_prompt_template.template.format(
        research_call=research_call,
        information_about_ncems=ncems_info,
        num=NUM_IDEAS_PER_MODEL,
    )
    print('✓ Idea prompt template prepared')
    print(f'✓ Configured to generate {NUM_IDEAS_PER_MODEL} ideas per model')
    print(f'\nPrompt preview (first 500 chars):\n{ideas_prompt[:500]}...')
else:
    print('Skipping idea generation prompt setup (GENERATE_NEW_IDEAS=False).')


In [ ]:
if GENERATE_NEW_IDEAS:
    # Generate ideas from each model
    all_ideas = []

    for model_name in models_to_use:
        logger.info(f"\n{'='*60}")
        logger.info(f"Generating {NUM_IDEAS_PER_MODEL} research ideas using {model_name}...")
        logger.info(f"{'='*60}")

        try:
            response = ai_interface.generate_content(
                prompt=ideas_prompt,
                model_name=model_name,
                temperature=0.7,
                max_completion_tokens=16000,
            )

            try:
                response_text = response.strip()
                start_idx = response_text.find('{')
                end_idx = response_text.rfind('}') + 1

                if start_idx != -1 and end_idx > start_idx:
                    json_str = response_text[start_idx:end_idx]
                    ideas_data = json.loads(json_str)

                    if 'research_ideas' in ideas_data:
                        research_ideas = ideas_data['research_ideas']
                        logger.info(f"✓ Generated {len(research_ideas)} ideas from {model_name}")

                        for idea in research_ideas:
                            idea['model'] = model_name
                            idea['author'] = model_name
                            idea['generated_at'] = datetime.now().isoformat()
                            all_ideas.append(idea)
                    else:
                        logger.error(f"No 'research_ideas' key found in response from {model_name}")
                else:
                    logger.error(f"Could not find valid JSON in response from {model_name}")

            except json.JSONDecodeError as e:
                logger.error(f"Failed to parse JSON from {model_name}: {e}")
                logger.error(f"Response preview: {response[:500]}...")

        except Exception as e:
            logger.error(f"Error generating ideas with {model_name}: {e}")

    print(f"\n{'='*60}")
    print(f"✓ Total ideas generated: {len(all_ideas)}")
    print(f"{'='*60}")
else:
    ideas_df = load_ideas_for_condition()


In [ ]:
if GENERATE_NEW_IDEAS:
    ideas_df = pd.DataFrame(all_ideas)

# Ensure required columns exist for downstream proposal generation
for col in ['model', 'author', 'title', 'abstract', 'generated_at']:
    if col not in ideas_df.columns:
        ideas_df[col] = ''

ideas_df['model'] = ideas_df['model'].apply(canonicalize_model)
ideas_df['author'] = ideas_df['author'].fillna(ideas_df['model'])

# Keep a canonical column order
canonical_cols = ['model', 'author', 'title', 'abstract', 'generated_at']
extra_cols = [c for c in ideas_df.columns if c not in canonical_cols]
ideas_df = ideas_df[canonical_cols + extra_cols]

# Drop rows missing essential fields
ideas_df = ideas_df.dropna(subset=['model', 'title', 'abstract']).copy()
ideas_df['title'] = ideas_df['title'].astype(str).str.strip()
ideas_df['abstract'] = ideas_df['abstract'].astype(str).str.strip()
ideas_df = ideas_df[(ideas_df['model'].astype(str).str.strip() != '') &
                    (ideas_df['title'] != '') &
                    (ideas_df['abstract'] != '')].reset_index(drop=True)

print(f"\nDataFrame shape: {ideas_df.shape}")
print(f"\nColumns: {list(ideas_df.columns)}")
print("\nIdeas per model:")
print(ideas_df['model'].value_counts(dropna=False))
print("\nFirst few rows:")
display(ideas_df.head())


In [ ]:
# Save a standardized ideas snapshot for this run
ideas_file = output_dir / f'ai_ideas_{condition_slug}_{timestamp}.csv'
ideas_df.to_csv(ideas_file, index=False)

print(f"✓ Saved standardized ideas table ({len(ideas_df)} rows) to: {ideas_file}")


## Step 2: Generate Full Proposals from Ideas

For each idea, use the same AI model to generate a comprehensive research proposal using the `generate_proposals` prompt.

In [ ]:
# Get the proposal-generation template
proposals_template = prompt_manager.get_template(PROPOSAL_PROMPT_TEMPLATE)

print(f"✓ Loaded proposal template: {PROPOSAL_PROMPT_TEMPLATE}")
print(f"Template parameters: {proposals_template.parameters}")


In [ ]:
# Add columns for full proposal sections (if missing)
proposal_section_cols = [
    'background_and_significance',
    'research_questions_and_hypotheses',
    'methods_and_approach',
    'expected_outcomes_and_impact',
    'open_science_and_reproducibility',
    'budget_and_resources',
]

for col in proposal_section_cols:
    if col not in ideas_df.columns:
        ideas_df[col] = ''
if 'proposal_generated_at' not in ideas_df.columns:
    ideas_df['proposal_generated_at'] = ''

print('✓ Proposal section columns ready')
print([c for c in proposal_section_cols + ['proposal_generated_at'] if c in ideas_df.columns])


In [ ]:
section_cols = [
    'background_and_significance',
    'research_questions_and_hypotheses',
    'methods_and_approach',
    'expected_outcomes_and_impact',
    'open_science_and_reproducibility',
    'budget_and_resources',
]

progress_files = sorted(output_dir.glob(f'ai_proposals_{condition_slug}_progress_*.csv'))
if progress_files:
    latest = progress_files[-1]
    ideas_df = pd.read_csv(latest)
    print(f"Loaded progress file: {latest.name}  ({len(ideas_df)} rows)")
elif 'ideas_df' not in globals():
    raise RuntimeError(
        'No progress files found and ideas_df is not defined. '
        'Run earlier cells to prepare ideas_df first.'
    )
else:
    print(f"No progress file found — using ideas_df from memory ({len(ideas_df)} rows).")

for col in section_cols:
    if col not in ideas_df.columns:
        ideas_df[col] = ''
if 'proposal_generated_at' not in ideas_df.columns:
    ideas_df['proposal_generated_at'] = ''


def _is_complete(row):
    model_val = row.get('model', '')
    if pd.isna(model_val) or str(model_val).strip() == '':
        return True  # skip malformed rows
    for c in section_cols:
        val = row.get(c, '')
        if pd.isna(val) or str(val).strip() == '':
            return False
    return True

already_done = sum(_is_complete(r) for _, r in ideas_df.iterrows())
print(f"Proposals already complete : {already_done} / {len(ideas_df)}")
print(f"Remaining to generate      : {len(ideas_df) - already_done}\n")

for idx, row in ideas_df.iterrows():
    if _is_complete(row):
        logger.info(f"Skipping {idx+1}/{len(ideas_df)} — already complete: {str(row.get('title',''))[:60]}")
        continue

    model_name = canonicalize_model(row.get('model', ''))
    title = str(row.get('title', '')).strip()
    abstract = str(row.get('abstract', '')).strip()
    author = str(row.get('author', row.get('model', '')))

    if not model_name or model_name not in available_models:
        logger.error(f"Skipping row {idx}: unavailable model '{model_name}'.")
        continue

    logger.info(f"\n{'='*60}")
    logger.info(f"Generating proposal {idx+1}/{len(ideas_df)} using {model_name}")
    logger.info(f"Author label: {author}")
    logger.info(f"Title: {title[:100]}...")
    logger.info(f"{'='*60}")

    try:
        proposal_prompt = proposals_template.template.format(
            research_call=research_call,
            information_about_ncems=ncems_info,
            title=title,
            abstract=abstract,
        )

        response = ai_interface.generate_content(
            prompt=proposal_prompt,
            model_name=model_name,
            temperature=0.7,
            max_completion_tokens=16000,
        )

        try:
            response_text = response.strip()
            start_idx = response_text.find('{')
            end_idx = response_text.rfind('}') + 1

            if start_idx != -1 and end_idx > start_idx:
                json_str = response_text[start_idx:end_idx]
                proposal_data = json.loads(json_str)
                proposal = proposal_data['proposal'] if 'proposal' in proposal_data else proposal_data

                ideas_df.at[idx, 'background_and_significance'] = proposal.get('background_and_significance', '')
                ideas_df.at[idx, 'research_questions_and_hypotheses'] = proposal.get('research_questions_and_hypotheses', '')
                ideas_df.at[idx, 'methods_and_approach'] = proposal.get('methods_and_approach', '')
                ideas_df.at[idx, 'expected_outcomes_and_impact'] = proposal.get('expected_outcomes_and_impact', '')
                ideas_df.at[idx, 'open_science_and_reproducibility'] = proposal.get('open_science_and_reproducibility', '')
                ideas_df.at[idx, 'budget_and_resources'] = proposal.get('budget_and_resources', '')
                ideas_df.at[idx, 'proposal_generated_at'] = datetime.now().isoformat()

                logger.info(f"✓ Successfully generated proposal for: {title[:60]}...")
            else:
                logger.error(f"Could not find valid JSON in response for: {title[:60]}...")

        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse JSON for '{title[:60]}...': {e}")
            logger.error(f"Response preview: {response[:500]}...")

    except Exception as e:
        logger.error(f"Error generating proposal for '{title[:60]}...': {e}")

    if (idx + 1) % 5 == 0:
        progress_file = output_dir / f'ai_proposals_{condition_slug}_progress_{timestamp}.csv'
        ideas_df.to_csv(progress_file, index=False)
        logger.info(f"✓ Progress saved to: {progress_file}")

print(f"\n{'='*60}")
print('✓ Completed proposal generation pass')
print(f"{'='*60}")


In [ ]:
# Save final results
final_file = output_dir / f'ai_proposals_{condition_slug}_complete_{timestamp}.csv'
ideas_df.to_csv(final_file, index=False)

print(f"✓ Saved complete proposals to: {final_file}")
print(f"\nFinal DataFrame shape: {ideas_df.shape}")
print("\nProposals with all sections completed:")
print(ideas_df['background_and_significance'].notna().sum())


## Summary Statistics

In [ ]:
# Display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\nCondition: {CONDITION}")
print(f"Total ideas/proposals rows: {len(ideas_df)}")
print("\nRows per model:")
print(ideas_df['model'].value_counts(dropna=False))
if 'author' in ideas_df.columns:
    print("\nTop author labels:")
    print(ideas_df['author'].fillna('').value_counts().head(10))

print("\nProposals with completed sections:")
for col in ['background_and_significance', 'research_questions_and_hypotheses',
            'methods_and_approach', 'expected_outcomes_and_impact',
            'open_science_and_reproducibility', 'budget_and_resources']:
    completed = ideas_df[col].notna().sum()
    print(f"  {col}: {completed}/{len(ideas_df)} ({completed/len(ideas_df)*100:.1f}%)")

print(f"\nAverage abstract length: {ideas_df['abstract'].astype(str).str.len().mean():.0f} characters")
print(f"Output files saved in: {output_dir}")
print("\n" + "="*60)


In [ ]:
# Display a sample proposal
print("\n" + "="*60)
print("SAMPLE PROPOSAL")
print("="*60)

sample_idx = 0
sample = ideas_df.iloc[sample_idx]

print(f"\nModel: {sample['model']}")
if 'author' in ideas_df.columns:
    print(f"Author label: {sample['author']}")
print(f"\nTitle: {sample['title']}")
print(f"\nAbstract: {str(sample['abstract'])[:300]}...")
print(f"\nBackground (first 300 chars): {str(sample['background_and_significance'])[:300]}...")
print("\n" + "="*60)
